# Least Squares

With more equations than unknowns there is usually no exact solution; least squares finds the one with the smallest sum of squared residuals. This notebook fits a polynomial to simulated noisy data three ways and shows why QR, the method behind `A \ b`, is the default.

In [ ]:
Pollis.packages("Plots")

In [ ]:
using LinearAlgebra, Plots

In [ ]:
using Random
rng = Xoshiro(1)   # fixed seed: the same data on every run
n = 50
t = sort(4 .* rand(rng, n))
truth(s) = 1 + 2s - 0.5s^2
y = truth.(t) .+ 0.5 .* randn(rng, n)
scatter(t, y, label="data", xlabel="t", ylabel="y")

## The design matrix

A quadratic $c_0 + c_1 t + c_2 t^2$ is linear in its coefficients. Each row of the design matrix holds $1, t_i, t_i^2$; the problem is $\min_c \lVert y - Ac \rVert$.

In [ ]:
A = [s^j for s in t, j in 0:2]
c = A \ y   # tall A: least squares by a pivoted QR
r = y - A * c
(coefficients = c, true_coefficients = [1.0, 2.0, -0.5], residual_norm = norm(r))

In [ ]:
ts = range(0, 4, length=200)
plot!(ts, [c[1] + c[2] * s + c[3] * s^2 for s in ts], lw=2, label="least-squares fit")
plot!(ts, truth.(ts), lw=1.5, ls=:dash, c=:black, label="true curve")

At the solution the residual is orthogonal to every column of $A$: $A^\top r = 0$, the normal equations.

In [ ]:
transpose(A) * r

## Three methods

QR works with $A$ directly. The normal equations form $A^\top A$, which squares the condition number. The SVD pseudo-inverse is the most robust and handles a rank-deficient $A$.

In [ ]:
c_qr = A \ y
c_ne = (transpose(A) * A) \ (transpose(A) * y)
c_svd = pinv(A) * y
[(method = m, coefficients = round.(v, digits=8), condition_faced = k) for (m, v, k) in (("QR", c_qr, cond(A)), ("normal equations", c_ne, cond(transpose(A) * A)), ("SVD", c_svd, cond(A)))]

Here the problem is well conditioned and the three agree to eight decimals. Raise the degree and the Vandermonde columns $1, t, t^2, \dots$ become nearly parallel: the normal equations lose digits much sooner.

In [ ]:
[(degree = d, cond_A = cond([s^j for s in t, j in 0:d]), cond_normal_equations = cond(transpose([s^j for s in t, j in 0:d]) * [s^j for s in t, j in 0:d])) for d in (2, 5, 8, 11)]

## Too many terms

More terms always lower the residual on the data. Past the true degree they fit the noise, and the fit goes astray outside the data.

In [ ]:
p = scatter(t, y, label="data", xlabel="t", ylabel="y", xlims=(0, 5), ylims=(-3, 5))
tw = range(0, 5, length=300)
for d in (2, 8)
    Ad = [s^j for s in t, j in 0:d]
    cfit = Ad \ y
    plot!(p, tw, [sum(cfit[j + 1] * s^j for j in 0:d) for s in tw], lw=2, label="degree $d")
end
plot!(p, tw, truth.(tw), ls=:dash, c=:black, label="true curve")
vline!(p, [4], c=:gray, ls=:dot, label="end of the data")

## References

[Trefethen and Bau (1997)](https://doi.org/10.1137/1.9780898719574), *Numerical Linear Algebra*, SIAM, Lectures 11 and 18–19.